# Lab 2 Tutorial — GANs and Diffusion Models
### A Beginner's Guide to Image Generation with PyTorch

---

Welcome! This tutorial walks you through every concept and piece of code in Lab 2,
step by step, in plain English. No prior experience with GANs or Diffusion models is assumed.

**What you will learn:**

| Section | Topic |
|---|---|
| 0 | Project structure and how all the pieces fit together |
| 1 | What is a GAN? — Vanilla GAN with BCE loss |
| 2 | Alternative loss — Logistic (non-saturating) loss |
| 3 | Conditional GAN — generating a specific digit |
| 4 | Adversarial attacks — fooling a CNN classifier |
| 5 | Diffusion models — how DDPM works from scratch |
| 6 | Theory Task — Ethical assessment of generative AI models |
| 7 | How to run the experiments |

> **How to use this tutorial:**  
> Read the explanation in each markdown cell, then look at the code cell below it.  
> You do **not** need to run this notebook — it is a reading guide. Run `main.ipynb` to execute experiments.

---
## Section 0 — Project Structure

Before we look at any model, let's understand how the code is organised.

```
Lab2/
├── config.py              ← All hyperparameters and paths live here
├── main.ipynb             ← The only notebook — calls everything below
│
├── data/
│   └── mnist_loader.py    ← Downloads and loads the MNIST dataset
│
├── models/
│   ├── vanilla_gan.py     ← Generator + Discriminator (Tasks 1 & 2)
│   ├── cgan.py            ← Conditional Generator + Discriminator (Task 3)
│   ├── cnn_classifier.py  ← CNN for digit classification (Task 4)
│   └── diffusion.py       ← Denoiser + GaussianDiffusion (Task 5)
│
├── training/
│   ├── gan_trainer.py     ← One-epoch training loop for GAN / cGAN
│   ├── cnn_trainer.py     ← Training + evaluation loop for CNN
│   └── diffusion_trainer.py ← Training loop for diffusion model
│
└── utils/
    ├── losses.py          ← BCE and Logistic loss functions
    ├── adversarial.py     ← FGSM attack + random noise classifier
    └── visualization.py   ← Saves image grids to disk
```

**Why is it split up like this?**  
This is called *separation of concerns*. Each file does exactly one job:
- `models/` only defines the network architectures — no training code.
- `training/` only contains training loops — no model definitions.
- `config.py` holds all numbers so you never have to search inside functions to change them.
- `main.ipynb` is the entry point — it just connects everything together.

This makes the code easy to read, test, and modify.

### 0.1 — config.py explained

`config.py` is the single place where all settings live.
If you want to change the number of training epochs or the learning rate, you only edit this file.

In [ ]:
# config.py — key settings (read-only in this tutorial)
import torch
from pathlib import Path

PROJECT_ROOT   = Path(".")          # Lab2/ folder
MNIST_ROOT     = str(PROJECT_ROOT.parent / "MNIST")   # shared dataset cache
CHECKPOINT_DIR = str(PROJECT_ROOT / "checkpoints")    # where .pth files are saved
OUTPUT_DIR     = str(PROJECT_ROOT / "outputs")        # where PNG grids are saved

# Automatically use GPU if available, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GAN_Z_DIM = 100   # latent noise vector size fed to the Generator
GAN_H_DIM = 256   # width of the hidden layer in Generator and Discriminator
GAN_X_DIM = 784   # 28 × 28 = 784 pixels per MNIST image (flattened)
GAN_LR    = 2e-4  # learning rate for both Generator and Discriminator

print(f"Running on: {DEVICE}")

### 0.2 — MNIST data loading

MNIST is a dataset of 70,000 handwritten digit images (0–9), each 28×28 pixels in greyscale.  
It is the "Hello World" of image machine learning.

We have two loaders with different transforms:

| Loader | Output shape | Value range | Used by |
|---|---|---|---|
| `get_flat_loader` | `(784,)` flat vector | `[0, 1]` | GAN, cGAN, Diffusion |
| `get_image_loader` | `(1, 28, 28)` image tensor | `[-1, 1]` | CNN classifier |

The CNN needs a 2-D spatial image so it can use its convolutional filters.  
The GAN and Diffusion models work with flat vectors — they don't use convolutions.

In [ ]:
# data/mnist_loader.py — what the loaders look like internally
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# For GAN / Diffusion: flatten to a 784-dim vector with values in [0, 1]
_FLAT_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),   # (1,28,28) → (784,)
])

# For CNN: keep 2-D shape, normalise to [-1, 1]
_IMAGE_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),      # shifts [0,1] to [-1,1]
])

---
## Section 1 — Task 1: Vanilla GAN with BCE Loss

### What is a GAN?

A **Generative Adversarial Network (GAN)** was introduced by Ian Goodfellow in 2014.  
The idea is to train two neural networks that compete against each other:

```
Random noise z  →  [Generator G]  →  Fake image
                                           ↓
Real image      →  [Discriminator D]  →  Real or Fake?
```

- **Generator (G):** Takes random noise as input and tries to produce images that look real.
- **Discriminator (D):** Takes an image (real or generated) and outputs a probability: how likely is it real?

They are trained simultaneously:
- **D is trained** to say "real" for real images and "fake" for generated ones.
- **G is trained** to fool D into saying "real" for its fake images.

Over time, G gets so good at faking that D can no longer tell the difference — at that point, G is generating realistic images.

### The Loss Functions (BCE)

We use **Binary Cross-Entropy (BCE)** loss.  
Think of it as: "how wrong was the prediction?"

$$\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1 - D(G(z)))]$$

In plain English:
- Punish D when it says a real image ($x$) is fake → `BCE(D(real), label=1)`
- Punish D when it says a fake image ($G(z)$) is real → `BCE(D(fake), label=0)`

$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$$

**Symbols in the GAN loss:**
- `\mathcal{L}_D` = discriminator loss.
- `\mathcal{L}_G` = generator loss.
- `\mathbb{E}[\cdot]` = expectation / average over the batch.
- `x` = a real image from the dataset.
- `z` = random noise vector input to the generator.
- `G(z)` = fake image produced by the generator.
- `D(x)` = discriminator probability that `x` is real.
- `D(G(z))` = discriminator probability that the generated image is real.

- Punish G when D correctly identifies its output as fake → `BCE(D(G(z)), label=1)`  
  (G wants D to output 1 = "real" for its fakes)

### 1.1 — Generator Architecture

In [ ]:
# models/vanilla_gan.py — Generator
import torch.nn as nn

class Generator(nn.Module):
    """
    Takes a random noise vector z of shape (batch, z_dim)
    and outputs a fake image of shape (batch, x_dim=784).

    z (100,) → Linear → ReLU → Linear → Sigmoid → x̂ (784,)

    Sigmoid at the end ensures output is in [0, 1],
    matching the [0, 1] range of real MNIST pixels.
    """
    def __init__(self, z_dim, h_dim, x_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, h_dim),   # 100 → 256
            nn.ReLU(),
            nn.Linear(h_dim, x_dim),   # 256 → 784
            nn.Sigmoid(),              # output in [0, 1]
        )

    def forward(self, z):
        return self.net(z)

### 1.2 — Discriminator Architecture

In [ ]:
# models/vanilla_gan.py — Discriminator

class Discriminator(nn.Module):
    """
    Takes a flat image (784,) and outputs a single number:
      use_sigmoid=True  → probability in [0, 1]  (Task 1 — BCE loss)
      use_sigmoid=False → raw logit              (Task 2 — Logistic loss)

    x (784,) → Linear → ReLU → Linear → [Sigmoid] → p (1,)
    """
    def __init__(self, x_dim, h_dim, use_sigmoid=True):
        super().__init__()
        layers = [
            nn.Linear(x_dim, h_dim),   # 784 → 256
            nn.ReLU(),
            nn.Linear(h_dim, 1),       # 256 → 1
        ]
        if use_sigmoid:
            layers.append(nn.Sigmoid())   # only for BCE (Task 1)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

### 1.3 — One Training Epoch

Each epoch loops over all batches of real images.  
For each batch, we do **two** gradient updates:

1. **Discriminator step** — update D to better tell real from fake.
2. **Generator step** — update G to better fool D.

Key detail: we call `.detach()` on the generated images during the D step.  
This stops PyTorch from computing gradients through G during D's update — we only want D's weights to change there.

In [ ]:
# training/gan_trainer.py — one training epoch (simplified for tutorial)
import torch

def train_vanilla_gan_epoch(G, D, G_opt, D_opt, loss_fn, loader, z_dim, device):
    G.train()
    D.train()
    d_total = g_total = 0.0

    for X_real, _ in loader:
        B      = X_real.size(0)              # batch size
        X_real = X_real.to(device)
        ones   = torch.ones(B, 1, device=device)   # label: real
        zeros  = torch.zeros(B, 1, device=device)  # label: fake

        # ── Step 1: Train Discriminator ──────────────────────────────── #
        z    = torch.randn(B, z_dim, device=device)  # random noise
        fake = G(z).detach()    # generate fake; detach = don't track G gradients

        D_loss = loss_fn(D(X_real), ones) + loss_fn(D(fake), zeros)
        D_opt.zero_grad()
        D_loss.backward()
        D_opt.step()

        # ── Step 2: Train Generator ──────────────────────────────────── #
        z      = torch.randn(B, z_dim, device=device)  # fresh noise
        G_loss = loss_fn(D(G(z)), ones)   # G wants D to output 1 (real)

        G_opt.zero_grad()
        G_loss.backward()
        G_opt.step()

        d_total += D_loss.item()
        g_total += G_loss.item()

    n = len(loader)
    return {"D_loss": d_total / n, "G_loss": g_total / n}

### 1.4 — Running Task 1 from main.ipynb

The notebook cell for Task 1 wires everything together:
- Create loader, loss function, models, and optimisers.
- Loop for 50 epochs, printing losses.
- Save a 4×4 grid of generated images at epochs 5, 10, and 50.
- Save model checkpoints to `checkpoints/`.

You will see output like:
```
Epoch   1/50  D_loss: 1.3862  G_loss: 0.6931
Epoch   5/50  D_loss: 0.8421  G_loss: 1.2034
  >> Sample grid saved: outputs/vanilla_gan/epoch_005.png
```

Early epochs will produce blurry/noisy images — this is expected with BCE loss. Read Task 2 to understand why.

---
## Section 2 — Task 2: Logistic (Non-Saturating) Loss

### The problem with BCE loss

Early in training, the Discriminator gets good quickly — it can easily tell apart real MNIST images from the Generator's early random-looking outputs.

When D is very confident, `D(G(z)) ≈ 0`.  
The Generator's BCE loss becomes:

$$\mathcal{L}_G = -\log(1 - D(G(z))) \approx -\log(1) = 0$$

The loss **saturates** near zero → the gradient is almost zero → **G stops learning**.

### The fix: Non-saturating (Logistic) loss

Instead of minimising `log(1 - D(G(z)))`, G is trained to maximise `log(D(G(z)))`:  

$$\mathcal{L}_G = -\log(\sigma(D(G(z))))$$

When D is confident (`D(G(z)) ≈ 0`), this loss is **large** not zero — so the gradient remains strong and G keeps learning.

### Implementation change

The Discriminator drops its `Sigmoid` and returns raw **logits** instead of probabilities.  
`BCEWithLogitsLoss` then applies Sigmoid internally (more numerically stable).

Only **two lines change** compared to Task 1:
```python
D = Discriminator(..., use_sigmoid=False)   # no Sigmoid on D
loss_fn = get_loss_fn("logistic")           # BCEWithLogitsLoss
```
Everything else — G architecture, training loop, epoch count — stays the same.

In [ ]:
# utils/losses.py — the two loss functions
import torch.nn.functional as F

def bce_loss(preds, targets):
    """Standard BCE. Requires Discriminator to use Sigmoid (Task 1)."""
    return F.binary_cross_entropy(preds, targets)

def logistic_loss(logits, targets):
    """Numerically stable logistic loss. Requires raw logits (Task 2)."""
    return F.binary_cross_entropy_with_logits(logits, targets)

def get_loss_fn(name):
    if name == "bce":      return bce_loss
    if name == "logistic": return logistic_loss
    raise ValueError(f"Unknown loss: {name}")

### Expected result

After running both tasks, `task2.view()` shows a side-by-side comparison:

| | Epoch 5 | Epoch 10 | Epoch 50 |
|---|---|---|---|
| BCE (Task 1) | Very blurry | Slightly better | Reasonable |
| Logistic (Task 2) | Clearer | Noticeably sharper | Good |

Logistic loss converges faster because the Generator always gets a meaningful gradient signal.

---
## Section 3 — Task 3: Conditional GAN (cGAN)

### The problem with vanilla GAN

The vanilla GAN generates a random digit — you have no control over which digit (0–9) it produces.

### The solution: conditioning on a label

A **Conditional GAN (cGAN)** feeds the class label `y` into both G and D as an extra input.

```
[Noise z  ‖  Embed(label y)]  →  Generator  →  Image of digit y
[Image x  ‖  Embed(label y)]  →  Discriminator  →  Is this a real image of digit y?
```

The label is turned into a learned vector via `nn.Embedding` — the same technique used in word embeddings in NLP.  
This vector is concatenated with the noise/image before the first Linear layer.

Now:
- G learns to generate images conditioned on a specific digit.
- D learns to distinguish real digit-y images from fake digit-y images.

At inference time, you pass the digit you want and get images of exactly that digit.

In [ ]:
# models/cgan.py — Conditional Generator
import torch
import torch.nn as nn

class ConditionalGenerator(nn.Module):
    """
    Inputs:
      z      — random noise (batch, z_dim)
      labels — class indices (batch,)  e.g. [3, 3, 3, ...]

    The label is looked up in an Embedding table → a learned vector of size embed_dim.
    This vector is concatenated with z before going through the network.
    """
    def __init__(self, z_dim, h_dim, x_dim, num_classes, embed_dim):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(z_dim + embed_dim, h_dim),  # z + label embedding
            nn.ReLU(),
            nn.Linear(h_dim, x_dim),
            nn.Sigmoid(),
        )

    def forward(self, z, labels):
        emb = self.label_emb(labels)          # (B, embed_dim)
        inp = torch.cat([z, emb], dim=1)      # (B, z_dim + embed_dim)
        return self.net(inp)

### What changes in the training loop?

Very little! The only difference vs vanilla GAN:
- Real images come with real labels → pass both to D: `D(X_real, y_real)`
- For the fake pass, G generates images for randomly sampled labels → `G(z, y_fake)`, then `D(fake, y_fake)`

This teaches the Discriminator to ask: "Is this a real **3**?" rather than just "Is this real?"

---
## Section 4 — Task 4: CNN Classifier + FGSM Adversarial Attack

### 4.1 — The CNN Classifier

Before we can attack a classifier, we need one. A **Convolutional Neural Network (CNN)** is well-suited for image classification because its convolutional filters can detect local patterns (edges, corners, curves) regardless of where they appear in the image.

Our CNN has two convolutional blocks followed by a fully-connected head:

```
Input (1, 28, 28)
  → Conv2d(1→32, 3×3) → ReLU → MaxPool(2×2)   → (32, 14, 14)
  → Conv2d(32→64, 3×3) → ReLU → MaxPool(2×2)  → (64, 7, 7)
  → Flatten                                    → (3136,)
  → Linear(3136→128) → ReLU → Dropout(0.25)
  → Linear(128→10)                             → logits for 10 classes
```

It achieves ~99% accuracy on MNIST after just 5 epochs.

In [ ]:
# models/cnn_classifier.py — CNN for digit classification
import torch.nn.functional as F

class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop  = nn.Dropout(p=0.25)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (B, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # (B, 64,  7,  7)
        x = x.view(x.size(0), -1)              # flatten to (B, 3136)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)                     # raw logits (B, 10)

### 4.2 — What is an Adversarial Attack?

An adversarial attack adds a tiny, carefully crafted perturbation to an image that causes a neural network to misclassify it — while the image still looks completely normal to a human eye.

### Fast Gradient Sign Method (FGSM)

FGSM was introduced by Goodfellow et al. (2014). It works by asking:

> "In which direction should I nudge each pixel to make the model more likely to predict the **target** class?"

That direction is the **gradient** of the loss (with respect to the input image), pointing toward the target class.  
We take a small step `ε` in that direction:

$$x_{\text{adv}} = \text{clip}\left(x - \varepsilon \cdot \text{sign}\left(\nabla_x \mathcal{L}(f(x), y_{\text{target}})\right)\right)$$

**Symbols in the FGSM equation:**
- $x_{\text{adv}}$ = adversarial image.
- $x$ = original input image.
- $\varepsilon$ = perturbation magnitude (attack strength).
- $\text{sign}(\cdot)$ = sign of the gradient, giving the direction for each pixel.
- $\nabla_x$ = gradient with respect to the input image.
- $\mathcal{L}(f(x), y_{\text{target}})$ = loss for the target class.
- $f(x)$ = model output for image $x$.
- $y_{\text{target}}$ = the wrong class we want the model to predict.

The **minus** sign is important — we are doing **targeted** attack (pushing toward a specific wrong class).  
An **untargeted** attack would use **plus** (push away from the true class).

In our experiment: source class = **4**, target class = **9** (make 4s look like 9s to the CNN).

In [ ]:
# utils/adversarial.py — FGSM targeted attack
import torch.nn.functional as F

def fgsm_targeted_attack(model, images, target_class, epsilon, device):
    model.eval()
    images = images.clone().detach().to(device)
    images.requires_grad_(True)   # we need gradients w.r.t. the IMAGE (not weights)

    targets = torch.full((images.size(0),), target_class,
                         dtype=torch.long, device=device)

    logits = model(images)
    loss   = F.cross_entropy(logits, targets)

    model.zero_grad()
    loss.backward()   # compute gradient of loss w.r.t. pixel values

    # Subtract gradient sign → move each pixel toward lower loss for target class
    adv_images = torch.clamp(
        images.detach() - epsilon * images.grad.sign(),
        min=-1.0, max=1.0,
    )
    return adv_images

### 4.3 — Random Noise Baseline

We also pass pure Gaussian noise (no digit at all) through the CNN.

**Why?** To show the difference between a *structured* adversarial perturbation (FGSM) and *random* noise:

- **FGSM:** The perturbation is computed from the model's gradients. Even though it's invisible to the human eye, it reliably fools the classifier because it exploits the model's exact decision boundary.
- **Random noise:** No structure aligned with any digit. The model still outputs *some* prediction (it has no "reject" option), but the results are random and change every run.

This demonstrates that adversarial examples are not just random noise — they are **adversarial** precisely because they target the model's weaknesses.

---
## Section 5 — Task 5: Conditional Diffusion Model (DDPM)

### The Big Idea

A **Denoising Diffusion Probabilistic Model (DDPM)** was introduced by Ho et al. (2020).  
The core idea is simple:

> **Train a neural network to reverse a noise-adding process.**

There are three parts:

---

### Step 1 — The Forward Process (destroying the image)

Start with a real MNIST image $x_0$.  
Add a tiny amount of Gaussian noise at each of $T = 1000$ steps.  
After 1000 steps, the image is pure random noise — all digit structure is gone.

The good news: we don't need to apply 1000 steps one by one.  
There's a shortcut formula to jump to any noise level $t$ directly:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

**Symbols in the forward diffusion equation:**
- $x_0$ = original clean image.
- $x_t$ = noisy image at timestep $t$.
- $\bar{\alpha}_t$ = cumulative noise schedule factor, controlling how much of the original image remains.
- $\varepsilon$ = Gaussian noise sample.
- $\mathcal{N}(0, I)$ = standard normal distribution with mean 0 and identity covariance.

- $\bar{\alpha}_t$ is close to 1 when $t$ is small (barely noisy) and close to 0 when $t = T$ (pure noise).
- This entire process requires **no learning** — it's just a fixed formula.

---

### Step 2 — Training the Denoiser

We train a neural network $\varepsilon_\theta(x_t, t, y)$ to **predict the noise** that was added.

Training loop for one batch:
1. Pick a real image $x_0$ and its label $y$.
2. Pick a random timestep $t \in \{1, \ldots, T\}$.
3. Sample noise $\varepsilon$ and compute $x_t$ using the shortcut formula above.
4. Ask the network: "What noise was added?"
5. Loss = MSE between the true noise $\varepsilon$ and the predicted noise.

$$\mathcal{L} = \|\varepsilon - \varepsilon_\theta(x_t, t, y)\|^2$$

Notice: no adversarial training, no competing networks — just a simple MSE loss. **Much easier to train than a GAN.**

The network also receives the timestep $t$ as a **sinusoidal embedding** so it knows how noisy the input is.

---

### Step 3 — Generating New Images (the Reverse Process)

To generate a new image of digit $y$:
1. Start from pure random noise $x_T \sim \mathcal{N}(0, I)$.
2. For $t = T, T-1, \ldots, 1$:
   - Ask the denoiser: "What noise is in this image?"
   - Subtract that noise (scaled appropriately) to get a slightly cleaner image.
   - Add a tiny bit of fresh noise for diversity (except at the final step).

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\varepsilon_\theta(x_t, t, y)\right) + \sqrt{\tilde{\beta}_t}\, z$$

**Symbols in the reverse diffusion step:**
- $x_{t-1}$ = the previous, less-noisy image.
- $\alpha_t$ and $\beta_t$ = fixed noise schedule coefficients at step $t$.
- $\tilde{\beta}_t$ = extra variance term used when adding a small amount of noise back in.
- $\varepsilon_\theta(x_t, t, y)$ = the model's predicted noise for the current noisy image, timestep, and class label.
- $z$ = fresh random Gaussian noise used in the reverse step.

3. After $T$ steps, you have a clean image of digit $y$.


In [ ]:
# models/diffusion.py — the denoiser network (simplified)

class ConditionalDenoiser(nn.Module):
    """
    MLP that takes (noisy_image, timestep, class_label) and predicts the noise.

    - Noisy image x_t    : flat vector (784,)
    - Timestep t         : encoded as a sinusoidal embedding (128,)
    - Class label y      : encoded via nn.Embedding           (128,)

    All three are concatenated and fed through 4 Linear layers.
    Output is the predicted noise (784,) — same shape as the image.
    """
    # (See models/diffusion.py for full implementation)
    pass

### Why does it work?

The network is trained on millions of examples: noisy images at every possible noise level, paired with the noise that was added.  
It builds a complete mental model of what every digit looks like at every noise level, and how to clean it up.

At inference, chaining 1000 tiny denoising steps together guides random noise gradually into a coherent digit image.

### GAN vs Diffusion — Summary

| | GAN (Tasks 1–3) | Diffusion (Task 5) |
|---|---|---|
| **Training** | Two networks competing — unstable, prone to collapse | Single MSE objective — stable and easy to scale |
| **Inference speed** | Single forward pass — very fast | T=1000 denoising steps — slow |
| **Image quality** | Good, but limited diversity | Higher quality and diversity |
| **Conditioning** | Embed label and concatenate | Same — works the same way |
| **Used in practice** | Older systems | DALL·E 2, Stable Diffusion, Imagen |

---
## Section 6 — Theory Task: Ethical Assessment of Generative AI Models

This is the **theoretical** part of the lab. It is separate from the coding tasks.

### Background

Models like **GPT-4, Stable Diffusion, DALL·E, Gemini, and Copilot** can generate remarkably realistic images. However, they have important limitations that are easy to miss if you don't look carefully:

- They are trained on large datasets that may not fully represent the real world.
- They learn statistical patterns — not physical or biological laws.
- They can produce images that look convincing at first glance but contain subtle errors.

### The Task

Use any generative image AI (GPT-4, Copilot, Gemini, etc.) to generate **at least 5 images of humans in real-life scenarios** — for example, people shaking hands, sitting at a desk, playing sport, or cooking.

Then **carefully inspect each image** and look for things that don't match reality.

> **Hint:** Pay close attention to **body composition** — fingers, hands, wrists, and limbs are common failure points.

### What to look for

| Category | Examples of inconsistencies |
|---|---|
| **Hands & fingers** | Extra or missing fingers, merged fingers, fingers bending the wrong way |
| **Arms & legs** | Unnatural proportions, extra limbs, joints bending backwards |
| **Faces** | Asymmetric eyes, distorted ears, teeth that don't look human |
| **Objects** | Text rendered incorrectly, impossible shadows, objects passing through each other |
| **Scene consistency** | Background items that contradict each other, incorrect reflections |

### Why does this happen?

Generative models do not "understand" human anatomy. They learn that images of people tend to have a certain number of fingers by seeing thousands of examples — but the statistics are noisy. When the model has to generate a detail (like a hand) that is partially occluded or in an unusual pose, it has fewer training examples to draw from and is more likely to hallucinate an impossible result.

This is a fundamental limitation of current **data-driven** (statistical) models:  
they interpolate and extrapolate from patterns in data rather than reasoning from first principles.

### What your written response should include

1. **A description of the inconsistency you found** — be specific (e.g., "the left hand has 6 fingers").
2. **Which model you used** and what prompt you gave it.
3. **Why you think it happened** — link it to the model's training or statistical nature.
4. **Optional:** Include a screenshot or image to illustrate your finding.

### Connection to this lab

In this lab you trained your own generative models (GAN, cGAN, Diffusion) on MNIST.  
Notice that even your best diffusion model sometimes generates slightly blurry or malformed digits — the same root cause as commercial models producing incorrect hands:  
the model has seen less data for unusual or edge-case examples, so it is less reliable there.

> This is why AI-generated content always needs **human review** before being used in any real-world context.

---
## Section 7 — How to Run the Experiments

Open `main.ipynb` and run the cells in order:

1. **Cell 1 (Setup)** — Always run this first. It checks your GPU, creates output folders, and imports all modules.
2. **Cell 2 (W&B — optional)** — Only run this if you want experiment tracking on wandb.ai.
3. **Task 1 cells** — Train vanilla GAN (BCE). Takes ~5 min on GPU, ~30 min on CPU.
4. **Task 2 cells** — Train vanilla GAN (Logistic). Same time. Then view the comparison.
5. **Task 3 cells** — Train cGAN. Then view the generated digits.
6. **Task 4 cells** — Train CNN (fast ~2 min), then run FGSM attack.
7. **Task 5 cells** — Train diffusion model (~20 epochs). **Slowest task.** Then compare with cGAN.

### Tips for beginners

- If you don't have a GPU, reduce `epochs` in `config.py` (e.g. set GAN epochs to 10) to see results faster.
- For diffusion, set `"timesteps": 200` in `config.py` for a much faster (lower quality) run.
- Checkpoints are saved to `checkpoints/` — if you re-run a cell, it will load from disk instead of retraining.
- All generated images are saved to `outputs/` — you can view them without running the view cells.

### Grading Checklist

| Grade | What you need |
|---|---|
| 3 | Run Tasks 1, 2, 3 — see generated digits — push code to GitHub |
| 4 | Also run Tasks 4 (adversarial attack) and 5 (diffusion) — needs GPU |
| 5 | Also be able to explain how the diffusion model works (see Section 5 above) + Theory Task 2.1 written assessment |